# SOLUTION - "PlantMate": Grounded Q&A over Real Word Documents

Reference solution for the PlantMate RAG project.

**Pipeline**

```
.docx files
   -> load_docx()        heading-aware text extraction
   -> chunk_document()   section-aware chunking with overlap
   -> embed_texts()      batched embeddings
   -> ChromaDB           cosine vector store + metadata
   -> search()           top-k + metadata filter + score threshold
   -> ask()              grounded generation with citations + refusal
   -> evaluate()         retrieval hit rate / answer accuracy / refusal accuracy
```

Run top to bottom. Every section maps 1:1 to a task in the problem statement.

---
## Step 0 - Environment, key, corpus

In [ ]:
# Colab / fresh environment setup
!pip install -q openai chromadb python-docx tiktoken

In [ ]:
# ---------------------------------------------------------------------------
# API KEY - NEVER hard-code a key inside a notebook you will share.
# A key pasted into a shared .ipynb is a leaked key and must be revoked.
# ---------------------------------------------------------------------------
import os, getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
print("client ready ->", CHAT_MODEL, "|", EMBEDDING_MODEL)

In [ ]:
# ============================================================================
# Builds the "real world" Word document corpus for this project.
# These are realistic corporate documents for a fictitious company,
# NorthWind Energy Services - an industrial / process plant services firm.
#
# >>> If you have your OWN .docx files, skip this cell, create a folder named
# >>> "company_docs" and drop your files in it. Everything downstream works
# >>> unchanged as long as your documents use real Word Heading styles.
# ============================================================================
import os
from docx import Document
from docx.shared import Pt

DOCS_DIR = "company_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

CORPUS = {
    'NorthWind_HSE_Policy_Manual.docx': [
        ('h1', 'NorthWind Energy Services - HSE Policy Manual'),
        ('p', 'Document No: HSE-MAN-001 | Revision: 4.2 | Effective Date: 01 April 2025 | Document Owner: Corporate HSE Department | Next Review: 01 April 2027'),
        ('h2', '1. Purpose and Scope'),
        ('p', 'This manual defines the minimum Health, Safety and Environment (HSE) requirements applicable to all NorthWind Energy Services employees, contractors, sub-contractors, trainees and visitors at every operating plant, project site, workshop and corporate office.'),
        ('p', 'Where a client HSE standard is more stringent than this manual, the client standard shall prevail. Any deviation from this manual requires written approval from the Corporate HSE Head.'),
        ('h2', '2. Permit to Work (PTW) System'),
        ('p', 'No non-routine work shall commence in a process area without a valid Permit to Work signed by the Area Authority and countersigned by the Performing Authority.'),
        ('p', 'A Hot Work Permit is valid for a maximum of 8 hours and for one shift only. It cannot be extended across shifts and must be re-issued after a shift change.'),
        ('p', 'A Confined Space Entry Permit is valid for a maximum of 12 hours. Entry is allowed only after a gas test confirms oxygen between 19.5% and 23.5% by volume, hydrocarbon below 5% LEL, and hydrogen sulphide below 5 ppm.'),
        ('p', 'Gas testing inside a confined space shall be repeated every 2 hours by a certified gas tester, and after any break in work exceeding 30 minutes. A trained standby person shall remain at the manway for the full duration of the entry.'),
        ('h2', '3. Personal Protective Equipment (PPE)'),
        ('p', 'The minimum PPE for entry into any process area is: flame retardant (FR) coverall, safety helmet with chin strap, safety goggles, safety shoes with steel toe cap, and hand gloves appropriate to the task.'),
        ('p', 'In designated H2S zones, a personal gas detector and an escape respirator (minimum 10 minutes rated) shall be carried at all times. Personnel working in H2S zones must hold a valid H2S awareness certificate renewed every 2 years.'),
        ('p', 'Face shields are mandatory for chemical handling, grinding and line breaking activities. Fall protection with a full body harness and double lanyard is mandatory for any work above 1.8 metres.'),
        ('h2', '4. Lockout Tagout (LOTO) and Energy Isolation'),
        ('p', 'Every energy isolation point shall carry an individual personal lock and a danger tag bearing the name, employee number and contact number of the person applying it.'),
        ('p', 'Where multiple trades work on the same equipment, a group lockout box shall be used. Each person places a personal lock on the group box and removes it only after their own work is complete.'),
        ('p', 'Only the person who applied a lock is authorised to remove it. Unauthorised removal of a lock or tag is treated as a Category 1 safety violation, resulting in disciplinary action and withdrawal of the site gate pass.'),
        ('h2', '5. Incident Reporting and Investigation'),
        ('p', 'All incidents, including near misses, first aid cases and process upsets, shall be reported verbally to the Shift In-Charge immediately and logged in the EHS Portal within 24 hours of occurrence.'),
        ('p', 'A Tier-1 process safety event, a lost time injury, or any fire requiring plant emergency response shall be notified to the Plant Manager and Corporate HSE within 2 hours.'),
        ('p', 'Root cause analysis for a Tier-1 event shall be completed within 15 working days using the 5-Why and Fishbone methodology, and shall be led by an investigator independent of the affected department.'),
        ('p', 'Each employee is expected to report a minimum of 5 near misses or unsafe conditions per calendar year. Reporting a near miss shall never attract punitive action.'),
        ('h2', '6. Emergency Response and Evacuation'),
        ('p', 'Plant alarm codes are as follows: a continuous siren of 2 minutes indicates a general emergency; an intermittent siren indicates a fire; a single continuous siren of 1 minute indicates all clear.'),
        ('p', 'On hearing the general emergency alarm, all personnel shall stop work, make equipment safe if it is safe to do so, and proceed upwind to the nearest assembly point A, B or C as marked on the plant layout.'),
        ('p', 'Head count at the assembly point shall be completed and reported to the Incident Controller within 10 minutes of the alarm. No person shall leave the assembly point without permission from the Assembly Point Warden.'),
        ('h2', '7. Contractor HSE Management'),
        ('p', 'Every contractor employee shall complete an 8-hour site HSE induction before being issued a gate pass. The induction is valid for 12 months and must be refreshed thereafter.'),
        ('p', 'Contractor HSE performance is scored monthly on leading and lagging indicators. A contractor scoring below 70% for two consecutive months shall be suspended from site until a corrective action plan is accepted.'),
        ('h2', '8. Environmental Compliance'),
        ('p', 'Stack emission monitoring shall be carried out monthly by an NABL accredited laboratory and the results submitted to the State Pollution Control Board quarterly.'),
        ('p', 'Treated effluent shall be maintained within a pH range of 6.5 to 8.5 before discharge. Any excursion beyond this range must be reported to the Environment Cell within 4 hours.'),
        ('p', 'Hazardous waste shall be stored in the designated covered yard for no longer than 90 days and disposed only through a CPCB authorised handler with a valid manifest.'),
    ],
    'NorthWind_Leave_and_Travel_Policy.docx': [
        ('h1', 'NorthWind Energy Services - Employee Leave and Business Travel Policy'),
        ('p', 'Document No: HR-POL-007 | Revision: 3.0 | Effective Date: 01 January 2025 | Document Owner: HR Operations | Applicable Payroll: India'),
        ('h2', '1. Applicability'),
        ('p', 'This policy applies to all confirmed full-time employees on the India payroll. Interns, retainers and third-party contract staff are governed by their individual engagement letters and are not covered by this policy.'),
        ('h2', '2. Leave Entitlements'),
        ('p', 'Earned Leave (EL): 21 working days per calendar year, accrued at 1.75 days per completed month of service. A maximum of 45 days of EL may be carried forward; any balance above 45 days lapses on 31 December. Accumulated EL is encashed at basic salary at the time of separation.'),
        ('p', 'Casual Leave (CL): 7 days per calendar year, to be availed for a maximum of 3 consecutive days at a time. Casual Leave cannot be carried forward and cannot be encashed.'),
        ('p', "Sick Leave (SL): 10 days per calendar year. A registered medical practitioner's certificate is mandatory for any sick leave exceeding 3 consecutive days."),
        ('p', 'Maternity Leave: 26 weeks of paid leave for the first two children and 12 weeks for the third child onwards, in line with the Maternity Benefit Act.'),
        ('p', 'Paternity Leave: 10 working days, to be availed within 6 months of the date of birth or adoption.'),
        ('p', 'Bereavement Leave: 5 calendar days in the event of the death of an immediate family member, defined as spouse, child, parent, sibling or parent-in-law.'),
        ('h2', '3. Leave Application and Approval'),
        ('p', 'All leave shall be applied through the Workday self-service portal. Earned Leave of more than 2 days must be applied at least 7 calendar days in advance.'),
        ('p', 'Emergency leave may be intimated by telephone or email to the reporting manager and must be regularised in Workday within 48 hours of resuming duty.'),
        ('p', 'Leave of up to 10 consecutive days is approved by the reporting manager. Any leave exceeding 10 consecutive days additionally requires Function Head approval.'),
        ('p', 'Leave without pay is permitted only after all leave balances are exhausted and requires HR Head approval.'),
        ('h2', '4. Site Deputation and Shift Allowances'),
        ('p', 'Employees deputed to a client site for a continuous period exceeding 15 days are eligible for a site deputation allowance of INR 1,200 per day from day 16 onwards.'),
        ('p', 'A night shift allowance of INR 450 per shift is payable for any shift where more than 4 hours fall between 22:00 and 06:00.'),
        ('p', 'Deputation allowance and per diem are not payable simultaneously for the same day; the higher of the two shall apply.'),
        ('h2', '5. Business Travel Entitlements'),
        ('p', 'Domestic air travel in economy class is permitted where the equivalent train journey exceeds 6 hours or the total door-to-door travel time exceeds 8 hours. In all other cases, AC 2-tier train travel is the default mode.'),
        ('p', 'Employees at Grade M3 and above may travel business class on international flights with a scheduled flying time exceeding 8 hours. All other international travel is in economy class.'),
        ('p', 'Hotel ceilings excluding taxes are INR 6,500 per night in metro cities and INR 4,500 per night in non-metro cities. Where a company or client guesthouse is available at the location, its use is mandatory.'),
        ('p', 'Airport transfers may be claimed on actuals against receipts. Use of the company travel desk for all bookings is mandatory; self-booked tickets require prior written approval.'),
        ('h2', '6. Per Diem'),
        ('p', 'Domestic per diem is INR 1,500 per day for metro cities and INR 1,100 per day for non-metro cities, payable without receipts to cover meals and incidentals.'),
        ('p', 'International per diem is USD 65 per day. Where meals are provided by the client or included in the hotel tariff, only 50% of the applicable per diem is payable.'),
        ('p', 'Per diem for the day of departure and the day of return is paid at 50% where the travel duration on that day is less than 8 hours.'),
        ('h2', '7. Expense Claims and Reimbursement'),
        ('p', 'All travel expense claims shall be submitted in Concur within 21 days of trip completion. Claims submitted after 45 days require written CFO approval and may be rejected.'),
        ('p', 'An original GST compliant invoice is mandatory for any single expense above INR 5,000. Expenses without supporting documents are limited to INR 500 per trip.'),
        ('p', 'Reimbursement is processed in the payroll cycle following claim approval, typically within 15 working days of manager approval.'),
        ('h2', '8. International Deputation and Visa'),
        ('p', 'Passport validity of at least 6 months beyond the intended date of return is mandatory before any international travel booking is confirmed.'),
        ('p', 'Corporate travel insurance is arranged by HR Operations for every international trip and is mandatory. Travel without an active policy number is not permitted.'),
        ('p', 'Any international deputation exceeding 90 days triggers a mandatory tax equalisation review by the Finance department before mobilisation.'),
    ],
    'NorthWind_Turnaround_Maintenance_SOP.docx': [
        ('h1', 'NorthWind Energy Services - Plant Turnaround and Shutdown Maintenance SOP'),
        ('p', 'Document No: SOP-MNT-014 | Revision: 2.1 | Effective Date: 15 June 2025 | Document Owner: Maintenance Head | Review Frequency: Every 2 years'),
        ('h2', '1. Purpose'),
        ('p', 'This Standard Operating Procedure defines the planning, execution, start-up and close-out requirements for a planned plant turnaround (TA) or major shutdown, to deliver the scope safely, on schedule and within approved cost.'),
        ('h2', '2. Turnaround Phases and Milestones'),
        ('p', 'Phase 1 - Long Range Planning begins at T minus 12 months and produces the preliminary scope register, budget estimate and TA organisation chart.'),
        ('p', 'Phase 2 - Detailed Planning begins at T minus 6 months and produces the level-3 schedule, resource histogram, long lead material order and contractor award.'),
        ('p', 'Phase 3 - Pre-Turnaround Execution begins at T minus 3 months and covers scaffolding, insulation removal, temporary facilities and pre-fabrication.'),
        ('p', 'Phase 4 - Execution covers shutdown, maintenance work and start-up. Phase 5 - Close-out is completed within 45 days of plant handover to Operations.'),
        ('h2', '3. Scope Management and Scope Freeze'),
        ('p', 'The turnaround scope shall be frozen at T minus 90 days. After the freeze date, no new scope may be added without written approval from the Turnaround Steering Committee.'),
        ('p', 'Any late scope addition estimated above 500 man-hours additionally requires joint approval from the Plant Manager and the Business Head, along with a schedule impact assessment.'),
        ('p', 'Every scope item shall carry a unique work order number, a justification category (statutory, reliability, or improvement) and an estimated man-hour value.'),
        ('h2', '4. Shutdown and Equipment Handover Sequence'),
        ('p', 'Throughput shall first be reduced to 50% of design capacity in a controlled manner, followed by feed cut as per the operations shutdown checklist.'),
        ('p', 'Equipment shall be depressurised to 0.5 barg, drained and purged with nitrogen until the hydrocarbon concentration is below 1% LEL, verified by two independent gas tests.'),
        ('p', 'Steam-out and water flushing shall follow purging where the service requires it. Isolation shall be completed strictly as per the approved blind list, and every blind installed shall be recorded in the blind register.'),
        ('p', 'Equipment is handed over to Maintenance only against a signed Equipment Handover Certificate issued by the Shift In-Charge, confirming isolation, de-energisation and gas-free status.'),
        ('h2', '5. Execution Control and Daily Reporting'),
        ('p', 'A daily turnaround coordination meeting shall be held at 07:00 hours, attended by the TA Manager, area coordinators, contractor supervisors and the HSE lead.'),
        ('p', 'Progress shall be tracked against a planned versus actual S-curve updated daily by 18:00 hours. Any critical path activity with float below 24 hours shall be escalated to the Turnaround Manager the same day.'),
        ('p', 'A recovery plan is mandatory when cumulative progress slips more than 5% behind plan.'),
        ('h2', '6. Quality Control and Inspection'),
        ('p', 'Hydrotest shall be carried out at 1.5 times the design pressure and held for a minimum of 30 minutes with no visible leakage or pressure drop.'),
        ('p', 'Positive Material Identification (PMI) is mandatory on 100% of alloy piping components before installation.'),
        ('p', 'Radiography coverage shall be 10% random for carbon steel welds and 100% for alloy welds. Any weld repair rate exceeding 5% triggers a welder qualification review.'),
        ('h2', '7. Start-up and Pre-Start-up Safety Review'),
        ('p', 'A Pre-Start-up Safety Review (PSSR) is mandatory before the introduction of hydrocarbons into any system, and shall be signed off by Operations, Maintenance, Inspection and HSE.'),
        ('p', 'All Punch List A items, defined as items that prevent safe start-up, must be closed to zero before PSSR sign-off. Punch List B items shall be closed within 30 days of start-up.'),
        ('p', 'Leak test at operating pressure shall be performed with nitrogen or with the process fluid at reduced pressure before ramping to full throughput.'),
        ('h2', '8. Close-out and Lessons Learned'),
        ('p', 'The turnaround close-out report shall be issued within 45 days of plant handover and shall include cost, schedule, HSE and quality performance against plan.'),
        ('p', 'A cost variance exceeding 10% of the approved budget requires a written explanation to the Business Head and the Finance Controller.'),
        ('p', 'Lessons learned shall be captured in a structured workshop within 30 days of start-up and uploaded to the corporate maintenance knowledge repository for reference in the next turnaround cycle.'),
    ],
}

for filename, blocks in CORPUS.items():
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)
    for kind, text in blocks:
        if kind == "h1":
            doc.add_heading(text, level=1)
        elif kind == "h2":
            doc.add_heading(text, level=2)
        else:
            doc.add_paragraph(text)
    doc.save(os.path.join(DOCS_DIR, filename))
    print("written:", filename)

print()
print("Corpus ready in ./%s" % DOCS_DIR)

### Baseline - the model without documents

Note how it answers: either a refusal, or a *plausible but invented* number.
That second case is exactly what RAG has to eliminate.

In [ ]:
# Baseline: ask the model WITHOUT any documents.
# This is the "why do we need RAG at all" cell - keep the output, you will
# contrast it with your RAG answer later.
baseline_questions = [
    "How long is a hot work permit valid at our plant?",
    "How many earned leave days do I get per year and how many can I carry forward?",
    "When is the turnaround scope freeze and who approves late additions?",
]

for q in baseline_questions:
    r = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": q}],
    )
    print("Q:", q)
    print("A:", r.choices[0].message.content.strip()[:400])
    print("-" * 90)

---
## Task 1 - Load the Word documents

`python-docx` exposes each paragraph's style name. Real Word headings come
through as `Heading 1`, `Heading 2`, ... We convert them to markdown markers so
the chunker downstream has an unambiguous, cheap signal for "a new section
starts here".

This is the step people get wrong by dumping the document into one flat string -
the section context is lost, and with it the ability to cite where an answer
came from.

In [ ]:
import os, glob, statistics
from docx import Document


def load_docx(path):
    """Extract text from a .docx, converting Word heading styles to markdown markers."""
    doc = Document(path)
    lines = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        style = (para.style.name or "").lower()
        if style.startswith("heading 1") or style == "title":
            lines.append("# " + text)
        elif style.startswith("heading 2"):
            lines.append("## " + text)
        elif style.startswith("heading"):
            lines.append("### " + text)
        else:
            lines.append(text)

    # Word tables carry real content too - flatten them into readable lines.
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells]
            if any(cells):
                lines.append(" | ".join(cells))

    return "\n".join(lines)


def load_corpus(folder="company_docs"):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*.docx"))):
        if os.path.basename(path).startswith("~$"):   # Word lock files
            continue
        docs.append({"title": os.path.basename(path), "text": load_docx(path)})
    return docs


documents = load_corpus()
print("documents loaded:", len(documents))
for d in documents:
    print("  %-48s %6d characters" % (d["title"], len(d["text"])))

In [ ]:
# Peek at the marked-up text so we can see the structure the chunker will use
print(documents[0]["text"][:900])

---
## Task 2 - Section-aware chunking with overlap

**Why 700 characters and 120 of overlap?**

The class demo used 350 characters with no overlap. That is too tight for these
documents: a single confined-space clause ("oxygen between 19.5% and 23.5%,
hydrocarbon below 5% LEL, hydrogen sulphide below 5 ppm") plus the sentence that
gives it context runs past 350 characters, so the limits get separated from the
activity they belong to and retrieval starts returning half an answer.

- **700 characters** (~170 tokens) holds a complete clause plus its neighbour,
  and still keeps embeddings sharp - a chunk that is too large gets a blurred
  vector that matches everything weakly and nothing strongly.
- **120 characters of overlap** means a fact that lands on a boundary appears in
  full in at least one chunk.
- **Never split across a heading.** A chunk that mixes "Per Diem" with
  "Expense Claims" produces a citation that is wrong even when the answer is right.

The splitter breaks on paragraph boundaries first, then sentence ends, and only
falls back to a hard cut if a single sentence is longer than `max_chars`.

In [ ]:
import re


def _split_long_text(text, max_chars, overlap):
    """Split text into <= max_chars pieces, preferring paragraph then sentence breaks."""
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    pieces, buffer = [], ""

    for para in paragraphs:
        # A paragraph that fits: append it to the running buffer.
        if len(buffer) + len(para) + 1 <= max_chars:
            buffer = (buffer + "\n" + para).strip()
            continue

        if buffer:
            pieces.append(buffer)
            buffer = ""

        # A single paragraph longer than max_chars -> split on sentence ends.
        if len(para) <= max_chars:
            buffer = para
        else:
            sentences = re.split(r"(?<=[.!?])\s+", para)
            for sent in sentences:
                if len(buffer) + len(sent) + 1 <= max_chars:
                    buffer = (buffer + " " + sent).strip()
                else:
                    if buffer:
                        pieces.append(buffer)
                    # Last resort: a single sentence longer than max_chars.
                    while len(sent) > max_chars:
                        pieces.append(sent[:max_chars])
                        sent = sent[max_chars - overlap:]
                    buffer = sent
    if buffer:
        pieces.append(buffer)

    # Apply overlap: prepend the tail of the previous piece to each piece.
    if overlap > 0 and len(pieces) > 1:
        overlapped = [pieces[0]]
        for i in range(1, len(pieces)):
            tail = pieces[i - 1][-overlap:]
            overlapped.append((tail + " " + pieces[i]).strip())
        pieces = overlapped

    return pieces


def chunk_document(doc, max_chars=700, overlap=120):
    """Heading-aware chunking. Every chunk belongs to exactly one section."""
    sections, current_heading, buffer = [], "Preamble", []

    for line in doc["text"].split("\n"):
        if line.startswith("#"):
            if buffer:
                sections.append((current_heading, "\n".join(buffer)))
                buffer = []
            current_heading = line.lstrip("#").strip()
        else:
            buffer.append(line)
    if buffer:
        sections.append((current_heading, "\n".join(buffer)))

    chunks = []
    for heading, body in sections:
        if not body.strip():
            continue
        for piece in _split_long_text(body, max_chars, overlap):
            chunks.append({
                "id": "%s::%03d" % (doc["title"], len(chunks)),
                "source": doc["title"],
                "section": heading,
                "chunk_index": len(chunks),
                # Prefixing the heading into the embedded text measurably improves
                # retrieval: the section name is often the vocabulary the user uses.
                "text": "[%s > %s]\n%s" % (doc["title"].replace(".docx", ""), heading, piece),
            })
    return chunks


chunks = []
for d in documents:
    chunks.extend(chunk_document(d))

lengths = [len(c["text"]) for c in chunks]
print("total chunks :", len(chunks))
print("min / median / max chars : %d / %d / %d"
      % (min(lengths), statistics.median(lengths), max(lengths)))
print()
for c in chunks[:3]:
    print("-" * 90)
    print(c["id"], "|", c["section"])
    print(c["text"][:300], "...")

---
## Task 3 - Embeddings and ChromaDB

Two things that bite people here:

1. **Batch the embedding calls.** One API call per chunk is slow and burns rate
   limit. We send 100 texts per request.
2. **Re-running the cell.** Chroma keeps the old collection alive in the session,
   so re-running `add()` duplicates vectors and quietly poisons retrieval. We
   delete the collection first, every time.

We use **cosine** space because OpenAI embeddings are direction-meaningful;
Chroma's default is L2, which is not what you want here.

In [ ]:
import chromadb


def embed_texts(texts, batch_size=100):
    """Batched embeddings -> list of vectors, order preserved."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        vectors.extend(item.embedding for item in response.data)
    return vectors


chroma_client = chromadb.Client()

COLLECTION_NAME = "plantmate"
try:
    chroma_client.delete_collection(COLLECTION_NAME)   # idempotent re-runs
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embed_texts([c["text"] for c in chunks]),
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"], "section": c["section"],
                "chunk_index": c["chunk_index"]} for c in chunks],
)

print("indexed chunks:", collection.count(), "| expected:", len(chunks))
assert collection.count() == len(chunks)

---
## Task 4 - Retrieval

`score = 1 - cosine_distance`, so higher is better and roughly in `[0, 1]` for
these embeddings.

`source` maps to a Chroma `where` filter - this is how you answer *"search only
the HSE manual"*, and it is also the cheapest accuracy win available when you
already know which document a question belongs to.

In [ ]:
def search(question, n=4, source=None, min_score=None):
    where = {"source": source} if source else None

    result = collection.query(
        query_embeddings=embed_texts([question]),
        n_results=n,
        where=where,
    )

    hits = []
    for meta, text, dist in zip(result["metadatas"][0],
                                result["documents"][0],
                                result["distances"][0]):
        score = 1 - dist
        if min_score is not None and score < min_score:
            continue
        hits.append({"source": meta["source"], "section": meta["section"],
                     "text": text, "score": score})
    return hits


def show(question, **kw):
    print("Q:", question)
    hits = search(question, **kw)
    if not hits:
        print("   (no hits above threshold)")
    for h in hits:
        print("   %.3f | %s > %s" % (h["score"], h["source"], h["section"]))
    print("-" * 95)


show("What PPE do I need in an H2S area?")
show("How much per diem do I get in Mumbai?")
show("What is the company policy on cryptocurrency trading?")   # nothing relevant exists

### Choosing the score threshold

Look at the third query above. Nothing in the corpus is about cryptocurrency,
yet Chroma still returns four chunks - **a vector store always returns its `n`
nearest neighbours, however far away they are**. Relevance is your job, not the
database's.

**Do not copy a threshold from a slide - read it off your own data.** The probe
cell below prints the top score for in-scope and out-of-scope questions. With
`text-embedding-3-small` on this corpus the two groups separate clearly; put your
threshold in the gap between them, nearer the bottom of the in-scope group than
the top of the out-of-scope group. `MIN_SCORE = 0.30` is the starting value used
below - **check the printed numbers and adjust it** before trusting the eval.

Set it too high and you refuse questions you could actually answer (worst kind of
failure - the information was right there). Too low and the model receives
irrelevant context and starts confabulating around it.

Re-tune this number whenever the corpus or the embedding model changes; it is a
property of your data, not a universal constant.

In [ ]:
MIN_SCORE = 0.30

# Evidence for the threshold: top score per question, in-scope vs out-of-scope
probe = [
    ("in-scope ", "What gas test readings are required for confined space entry?"),
    ("in-scope ", "How many earned leave days can I carry forward?"),
    ("in-scope ", "When is the turnaround scope freeze?"),
    ("out-scope", "What is the company policy on cryptocurrency trading?"),
    ("out-scope", "Who is the CEO of NorthWind Energy Services?"),
    ("out-scope", "What is the recipe for a chocolate cake?"),
]
for label, q in probe:
    top = search(q, n=1)[0]["score"]
    print("%s | top score %.3f | %s" % (label, top, q))

---
## Task 5 - Grounded generation with citations

Three defences against hallucination, in order of how much they buy you:

1. **Refuse before calling the model.** If nothing clears the threshold, there is
   no point asking the LLM - it can only invent. This is deterministic and free.
2. **A system prompt that forbids outside knowledge** and mandates a citation.
   Citations are not decoration: they make the answer auditable, and an answer
   that cannot be cited is usually an answer that was made up.
3. **`temperature=0`** for reproducibility. A policy assistant that gives a
   different number on Tuesday is worse than useless.

In [ ]:
SYSTEM_PROMPT = """You are PlantMate, an assistant that answers questions about
NorthWind Energy Services internal policy documents.

RULES
1. Answer ONLY from the numbered context passages provided. You have no other
   knowledge. Do not use anything you know about other companies or general practice.
2. After each factual claim, cite the passage it came from as [source > section].
3. Reproduce numbers, limits, durations and thresholds EXACTLY as written. Never
   estimate, round, convert or infer a number that is not in the context.
4. If the context does not contain the answer, reply with exactly:
   I don't know based on the documents provided.
   Do not apologise, do not speculate, do not offer general advice.
5. If the context answers only part of the question, answer that part and state
   plainly which part is not covered by the documents.
6. Be concise. Prefer short paragraphs or bullets."""


REFUSAL = "I don't know based on the documents provided."


def build_context(hits):
    blocks = []
    for i, h in enumerate(hits, start=1):
        blocks.append("[%d] source: %s | section: %s\n%s"
                      % (i, h["source"], h["section"], h["text"]))
    return "\n\n".join(blocks)


def ask(question, n=4, source=None, min_score=MIN_SCORE,
        show_context=False, verbose=True):
    """Retrieve, ground, answer. Returns (answer, hits)."""
    hits = search(question, n=n, source=source, min_score=min_score)

    # Defence 1: retrieval already failed - do not let the model improvise.
    if not hits:
        if verbose:
            print("Q:", question)
            print("A:", REFUSAL, "  (no chunk cleared the score threshold)")
            print("=" * 95)
        return REFUSAL, []

    user_prompt = ("Question: %s\n\nContext passages:\n%s"
                   % (question, build_context(hits)))

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_prompt}],
    )
    answer = response.choices[0].message.content.strip()

    if verbose:
        print("Q:", question)
        print("A:", answer)
        print("\nretrieved:")
        for h in hits:
            print("   %.3f | %s > %s" % (h["score"], h["source"], h["section"]))
        if show_context:
            print("\ncontext sent to the model:\n" + build_context(hits))
        print("=" * 95)

    return answer, hits

In [ ]:
_ = ask("What are the gas test limits before entering a confined space?")
_ = ask("How many earned leave days do I get and how many can I carry forward?")
_ = ask("When is the turnaround scope frozen and who approves late additions?")

In [ ]:
# Refusal behaviour - the questions the corpus cannot answer
_ = ask("What is the company policy on cryptocurrency trading by employees?")
_ = ask("Who is the Chief Executive Officer of NorthWind Energy Services?")

In [ ]:
# Metadata filtering - restrict the search to one document
_ = ask("What is the notification timeline for a serious event?",
        source="NorthWind_HSE_Policy_Manual.docx")

# Partial-answer behaviour: half of this is in the docs, half is not
_ = ask("What is the per diem in Mumbai, and what is the reimbursement rate for my own car?")

---
## Task 6 - Evaluation

Three metrics, each answering a different question:

- **Retrieval hit rate** - *did we even fetch the right document?* If this is
  low, no prompt engineering will save you; fix chunking or embeddings.
- **Answer accuracy** - *given the right context, did the model use it correctly?*
- **Refusal accuracy** - *does it keep quiet when it should?* Skipping this metric
  is how teams ship a confident liar.

`expect_keyword` is a crude proxy for correctness - it checks that the right
number appears, not that the sentence around it is right. It is honest enough for
a project and cheap enough to run on every change. In production you would grade
against reference answers with a human reviewer or an LLM judge, and track the
score over time as documents get revised.

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation set: 6 answerable questions + 2 deliberately unanswerable ones.
# "expect_source"  -> which document SHOULD be retrieved
# "expect_keyword" -> a string that must appear in a correct answer
# "answerable=False" -> the assistant MUST refuse instead of inventing an answer
# ---------------------------------------------------------------------------
EVAL_SET = [
    {"question": "What gas test readings are required before a confined space entry?",
     "expect_source": "NorthWind_HSE_Policy_Manual.docx", "expect_keyword": "19.5", "answerable": True},
    {"question": "How long is a hot work permit valid and can it be extended across shifts?",
     "expect_source": "NorthWind_HSE_Policy_Manual.docx", "expect_keyword": "8 hours", "answerable": True},
    {"question": "How many earned leave days do I get in a year and what is the carry forward limit?",
     "expect_source": "NorthWind_Leave_and_Travel_Policy.docx", "expect_keyword": "21", "answerable": True},
    {"question": "What is the per diem for domestic travel to a metro city?",
     "expect_source": "NorthWind_Leave_and_Travel_Policy.docx", "expect_keyword": "1,500", "answerable": True},
    {"question": "When is the turnaround scope frozen and who approves late scope additions?",
     "expect_source": "NorthWind_Turnaround_Maintenance_SOP.docx", "expect_keyword": "90", "answerable": True},
    {"question": "At what pressure is a hydrotest carried out and for how long is it held?",
     "expect_source": "NorthWind_Turnaround_Maintenance_SOP.docx", "expect_keyword": "1.5", "answerable": True},
    {"question": "What is the company policy on cryptocurrency trading by employees?",
     "expect_source": None, "expect_keyword": None, "answerable": False},
    {"question": "Who is the Chief Executive Officer of NorthWind Energy Services?",
     "expect_source": None, "expect_keyword": None, "answerable": False},
]

print(len(EVAL_SET), "evaluation questions loaded",
      "|", sum(1 for e in EVAL_SET if e["answerable"]), "answerable",
      "|", sum(1 for e in EVAL_SET if not e["answerable"]), "must-refuse")

In [ ]:
def evaluate(eval_set, n=4, min_score=MIN_SCORE):
    rows = []
    for item in eval_set:
        q = item["question"]
        hits = search(q, n=n, min_score=min_score)
        answer, _ = ask(q, n=n, min_score=min_score, verbose=False)

        if item["answerable"]:
            retrieved = item["expect_source"] in [h["source"] for h in hits]
            correct = item["expect_keyword"] in answer
            refused_ok = None
        else:
            retrieved = None
            correct = None
            refused_ok = REFUSAL.lower() in answer.lower()

        rows.append({"question": q, "answerable": item["answerable"],
                     "retrieved": retrieved, "correct": correct,
                     "refused_ok": refused_ok, "answer": answer,
                     "top_score": hits[0]["score"] if hits else 0.0})
    return rows


results = evaluate(EVAL_SET)

print("%-4s %-9s %-9s %-8s %-7s %s" % ("#", "type", "retrieved", "correct", "score", "question"))
print("-" * 110)
for i, r in enumerate(results, 1):
    kind = "answer" if r["answerable"] else "refuse"
    ret = {True: "yes", False: "NO", None: "-"}[r["retrieved"]]
    if r["answerable"]:
        ok = "yes" if r["correct"] else "NO"
    else:
        ok = "yes" if r["refused_ok"] else "NO"
    print("%-4d %-9s %-9s %-8s %-7.3f %s" % (i, kind, ret, ok, r["top_score"], r["question"][:52]))

answerable = [r for r in results if r["answerable"]]
unanswerable = [r for r in results if not r["answerable"]]

hit_rate = sum(r["retrieved"] for r in answerable) / len(answerable)
accuracy = sum(r["correct"] for r in answerable) / len(answerable)
refusal = sum(r["refused_ok"] for r in unanswerable) / len(unanswerable)

print()
print("Retrieval hit rate : %.0f%%  (%d/%d)" % (hit_rate * 100, sum(r["retrieved"] for r in answerable), len(answerable)))
print("Answer accuracy    : %.0f%%  (%d/%d)" % (accuracy * 100, sum(r["correct"] for r in answerable), len(answerable)))
print("Refusal accuracy   : %.0f%%  (%d/%d)" % (refusal * 100, sum(r["refused_ok"] for r in unanswerable), len(unanswerable)))

In [ ]:
# Inspect anything that failed - always read the retrieved chunks before "fixing the prompt"
for r in results:
    failed = (r["answerable"] and not (r["retrieved"] and r["correct"])) or \
             (not r["answerable"] and not r["refused_ok"])
    if failed:
        print("FAILED:", r["question"])
        print("answer:", r["answer"][:400])
        print("-" * 95)
else:
    print("(inspection loop complete)")

### Ablation - why the class-demo chunker was not enough

This rebuilds the index the way the live class did it (350 characters, no
overlap, no heading prefix, no threshold) and re-runs the same evaluation, so the
chunking argument is backed by a number instead of an opinion.

Run it and report the two lines it prints. The expectation is that the naive
configuration loses most on **refusal accuracy** (it has no score threshold, so it
never declines) and on questions whose limit and context got split apart - but
**report what you actually measure**, including the case where the difference is
small.

In [ ]:
def naive_chunks(doc, max_size=350):
    """The live-class chunker: fixed size, no overlap, no heading in the text."""
    out, section, buffer = [], "Introduction", ""
    for line in doc["text"].split("\n"):
        is_heading = line.startswith("#")
        is_full = len(buffer) + len(line) > max_size
        if buffer.strip() and (is_heading or is_full):
            out.append({"id": "%s#naive%03d" % (doc["title"], len(out)),
                        "source": doc["title"], "section": section,
                        "text": buffer.strip()})
            buffer = ""
        if is_heading:
            section = line.replace("#", "").strip()
        else:
            buffer += line + "\n"
    if buffer.strip():
        out.append({"id": "%s#naive%03d" % (doc["title"], len(out)),
                    "source": doc["title"], "section": section, "text": buffer.strip()})
    return out


baseline_chunks = []
for d in documents:
    baseline_chunks.extend(naive_chunks(d))

try:
    chroma_client.delete_collection("plantmate_naive")
except Exception:
    pass

naive_collection = chroma_client.create_collection(
    name="plantmate_naive", metadata={"hnsw:space": "cosine"})
naive_collection.add(
    ids=[c["id"] for c in baseline_chunks],
    embeddings=embed_texts([c["text"] for c in baseline_chunks]),
    documents=[c["text"] for c in baseline_chunks],
    metadatas=[{"source": c["source"], "section": c["section"]} for c in baseline_chunks],
)

# Swap the collection, re-run the identical evaluation, then swap back.
tuned_collection = collection
collection = naive_collection
naive_results = evaluate(EVAL_SET, min_score=0.0)   # class demo had no threshold
collection = tuned_collection

def summarise(rows, label):
    a = [r for r in rows if r["answerable"]]
    u = [r for r in rows if not r["answerable"]]
    print("%-28s hit-rate %3.0f%% | accuracy %3.0f%% | refusal %3.0f%%" % (
        label,
        100 * sum(r["retrieved"] for r in a) / len(a),
        100 * sum(r["correct"] for r in a) / len(a),
        100 * sum(r["refused_ok"] for r in u) / len(u)))

print("chunks: tuned=%d  naive=%d" % (len(chunks), len(baseline_chunks)))
print()
summarise(naive_results, "naive 350 / no overlap")
summarise(results,       "tuned 700 / 120 overlap")

---
## Task 7 - Analysis

**1. Chunk size.** 700 characters with 120 of overlap, splitting on paragraph then
sentence boundaries, never across a heading. The risk the 350-character no-overlap
chunker runs is separating a limit from the activity it governs - a clause can be
cut so that the gas limits sit in one chunk and the sentence naming confined-space
entry sits in another, and a question phrased around the activity then retrieves
only half the answer. Use the ablation cell above to show whether that actually
bites on this corpus, and cite the specific question if it does. Prefixing `[document > section]` into
the embedded text helps further, because users tend to phrase questions in the
vocabulary of the section title.

**2. Threshold.** `MIN_SCORE = 0.30` as a starting point, then read the probe cell
and move it into the gap between your in-scope and out-of-scope score bands -
**quote your actual numbers here.** Too high and genuinely answerable
questions get refused - the worst failure mode, because the information was in the
corpus and the user now believes it is not. Too low and irrelevant context reaches
the model, which will dutifully build an answer around it.

**3. Failures.** Read the retrieved chunks before touching the prompt. If
`retrieved` is `NO`, it is a retrieval problem - chunking, embedding, or `n`. If
`retrieved` is `yes` but `correct` is `NO`, the context was there and the model
mishandled it - that is a prompt or model problem. The two look identical from the
answer alone and have completely different fixes.

**4. Before / after.** Put the Step 0 answers next to the RAG answers for the same
three questions. The behaviour to look for in the no-RAG output is the mix of
honest refusal and *plausible general-industry number offered with equal
confidence* - the user cannot tell those two apart. The RAG answers should carry
the document's own figures plus a citation naming the section, which is what makes
them checkable.

**5. Before real employees use this:**

- **Document freshness.** Policies get revised. Store the revision number and
  effective date per chunk, surface them in the answer, and re-index automatically
  when a file changes. A confidently cited answer from revision 3.0 after 4.0 is
  published is worse than no answer.
- **Access control.** Retrieval must respect who is asking. Filter by the user's
  entitlements at query time - never rely on the prompt to keep a secret.
- **Logging and feedback.** Log every question, the retrieved chunk ids and the
  answer. This is your evaluation set for next quarter, and your audit trail when
  someone disputes an answer.
- **Evaluation cadence.** Run this eval on every document update and prompt change,
  not once at the end.
- **Human escalation.** For safety-critical questions, cite and link the source
  document and tell the user to confirm with the Area Authority. The assistant
  speeds up lookup; it does not carry the accountability.

---
## Bonus - hybrid retrieval

Pure vector search is weak on rare exact tokens: `SOP-MNT-014`, `PSSR`, `HSE-MAN-001`.
Embeddings capture meaning, and a document number has almost none - it is a string
that must match literally.

The fix is to blend a keyword score into the vector score. This runs the keyword
side in plain Python over the chunk list, which is fine at this corpus size; at
scale you would use BM25 or your database's full-text index.

In [ ]:
def keyword_score(question, text):
    """Fraction of the question's distinctive tokens that appear in the chunk."""
    tokens = {t for t in re.findall(r"[A-Za-z0-9\-]{3,}", question.lower())}
    if not tokens:
        return 0.0
    text_low = text.lower()
    return sum(1 for t in tokens if t in text_low) / len(tokens)


def hybrid_search(question, n=4, pool=12, alpha=0.75):
    """alpha weights the vector score; (1 - alpha) weights the keyword score."""
    candidates = search(question, n=pool)
    for c in candidates:
        c["hybrid"] = alpha * c["score"] + (1 - alpha) * keyword_score(question, c["text"])
    candidates.sort(key=lambda c: c["hybrid"], reverse=True)
    return candidates[:n]


for q in ["What does SOP-MNT-014 say about punch list items?",
          "Explain the PSSR requirement before start-up"]:
    print("Q:", q)
    print("  vector only:")
    for h in search(q, n=3):
        print("     %.3f | %s > %s" % (h["score"], h["source"], h["section"]))
    print("  hybrid:")
    for h in hybrid_search(q, n=3):
        print("     %.3f | %s > %s" % (h["hybrid"], h["source"], h["section"]))
    print("-" * 95)

---
## Bonus - a simple chat loop with follow-up support

A follow-up like *"and for non-metro?"* is meaningless to a retriever on its own -
there is nothing to embed. The standard fix is **query rewriting**: use the
conversation history to expand the follow-up into a standalone question, then
retrieve on that. Retrieval quality depends entirely on the rewrite, so it is worth
printing the rewritten query while you are developing.

In [ ]:
def rewrite_query(history, question):
    """Turn a context-dependent follow-up into a standalone question."""
    if not history:
        return question
    convo = "\n".join("%s: %s" % (role, text) for role, text in history[-4:])
    r = client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[
            {"role": "system", "content":
             "Rewrite the user's latest message as a standalone question using the "
             "conversation for context. Output only the rewritten question, nothing else."},
            {"role": "user", "content": "Conversation:\n%s\n\nLatest message: %s" % (convo, question)},
        ])
    return r.choices[0].message.content.strip()


def chat(questions):
    history = []
    for q in questions:
        standalone = rewrite_query(history, q)
        if standalone != q:
            print("(rewritten -> %s)" % standalone)
        answer, _ = ask(standalone, verbose=False)
        print("User:", q)
        print("PlantMate:", answer)
        print("-" * 95)
        history.append(("user", q))
        history.append(("assistant", answer))


chat([
    "What is the per diem for domestic travel to a metro city?",
    "And for non-metro?",
    "What if the client provides meals?",
])

---
## What to hand in

This notebook, run top to bottom, plus your own analysis cells. Check before submitting:

- [ ] No API key anywhere in the file
- [ ] `collection.count()` matches the chunk count
- [ ] All 8 eval questions executed with visible output
- [ ] The three metrics printed
- [ ] The naive-vs-tuned ablation run
- [ ] Analysis written in your own words